# Section 2b: Structural Variant — Oncogenic Gene Fusion Detection

Welcome to **Section 2b: Structural Variant - Oncogenic Gene Fusion Detection**. In this notebook, we cover the mechanisms of oncogenic gene fusions and detect chimeric transcripts using **`GeneFuse`**.

### Workflow Overview:
1. **Theoretical Foundations of Gene Fusions & Breakpoints**
2. **Hg38 Cancer & Druggable Fusion Panel Setup**
3. **Executing GeneFuse on Paired-End FASTQ Sequencing Data**
4. **Parsing & Visualizing Gene Fusion Call Reports (Pandas & JSON)**


---  
## Theoretical Background: Gene Fusions & Chimeric Transcripts

### 1. Mechanisms of Oncogenic Gene Fusions
Gene fusions occur when genomic structural rearrangements (translocations, inversions, or interstitial deletions) join coding sequences from two distinct genes, forming a hybrid chimeric protein with altered oncogenic function:

- **Kinase Fusions** (e.g. *EML4-ALK*, *KIF5B-RET*, *CD74-ROS1*, *NTRK1/2/3* fusions): Fusion of a constitutively active dimerization domain to a receptor tyrosine kinase domain, leading to ligand-independent kinase activation and uncontrolled cell proliferation.
- **Transcription Factor Fusions** (e.g. *BCR-ABL1*, *TMPRSS2-ERG*): Chimeric transcription factors driving aberrant expression of target oncogenes.

### 2. Sequence Read Evidence: Split Reads vs Discordant Pairs
- **Split Reads**: Individual sequencing reads that span the exact exon-exon junction of the gene fusion. The 5' portion aligns to Gene A and the 3' portion aligns to Gene B.
- **Discordant Paired-End Reads**: Read 1 aligns to Gene A while Read 2 aligns to Gene B on a different chromosome or beyond expected insert size limits.

### 3. K-Mer Junction Matching Algorithm (`GeneFuse`)
Rather than performing time-consuming genome-wide alignment of billions of reads, `GeneFuse` pre-indexes k-mer sequences from known cancer gene fusion exons (`cancer.hg38.csv`). It streams raw FASTQs directly, pinpointing split reads and fusion breakpoints in seconds.

> 💡 **Bioinformatics Interview Q&A: Gene Fusions & Split Reads**
>
> **Q: What is the technical distinction between Split Reads and Discordant Paired-End Reads in fusion detection, and how does GeneFuse achieve fast fusion discovery?**
>
> **A:** **Split Reads** physically cross the exact fusion breakpoint, where a single read's 5' end matches Gene A and its 3' end matches Gene B. **Discordant Paired-End Reads** have Read 1 mapping to Gene A and Read 2 mapping to Gene B on different chromosomes or orientations. Standard callers align reads to the whole genome first; `GeneFuse` pre-indexes k-mer sequences of candidate fusion junctions (`cancer.hg38.csv`) and streams FASTQs directly, identifying split reads and fusion junctions in seconds without full-genome alignment overhead.


---  
## Step 1: Run GeneFuse Fusion Calling

We execute `genefuse` using paired raw FASTQ reads (`SRR7890850_1.fastq.gz`, `SRR7890850_2.fastq.gz`), the Chromosome 17 reference FASTA (`chr17.fa`), and the hg38 cancer gene fusion library (`References/GeneFuse/cancer.hg38.csv`).


In [ ]:
%%bash
mkdir -p Results/Fusion
genefuse \
  -1 Samples/SRR7890850_1.fastq.gz \
  -2 Samples/SRR7890850_2.fastq.gz \
  -f References/GeneFuse/cancer.hg38.csv \
  -r References/chr17.fa \
  -h Results/Fusion/genefuse_report.html \
  -j Results/Fusion/genefuse_report.json \
  -t 4


---  
## Step 2: Parse & Inspect Gene Fusion Results (Pandas & JSON)

We load `genefuse_report.json` using Python and Pandas to analyze detected fusion events, breakpoint coordinates, supporting read counts, and fusion transcripts.


In [ ]:
import json
import pandas as pd
import os

json_report = "Results/Fusion/genefuse_report.json"
if os.path.exists(json_report):
    with open(json_report) as f:
        data = json.load(f)
        
    fusions_raw = data.get("fusions", [])
    if isinstance(fusions_raw, dict):
        fusions = list(fusions_raw.values())
    else:
        fusions = fusions_raw
    print(f"=== GeneFuse Execution Summary ===")
    print(f"Total Reads Processed: {data.get('total_reads', 'N/A')}")
    print(f"Total Candidate Fusions Identified: {len(fusions)}")
    
    if fusions:
        records = []
        for fus in fusions:
            records.append({
                "Fusion Name": fus.get("name", ""),
                "Gene 1": fus.get("gene1", ""),
                "Gene 2": fus.get("gene2", ""),
                "Breakpoint 1": fus.get("position1", ""),
                "Breakpoint 2": fus.get("position2", ""),
                "Unique Reads": fus.get("unique_reads", 0),
            })
        df_fus = pd.DataFrame(records)
        print("\n=== Identified Gene Fusion Events Table ===")
        display(df_fus)
    else:
        print("\nNo oncogenic gene fusions detected in the current sequencing sample.")
        print("Interactive HTML visualization saved to: Results/Fusion/genefuse_report.html")
